In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
%pip install --upgrade torch-geometric-signed-directed networkx

In [0]:
# dbutils.library.restartPython()

In [0]:
import os

# ============================================================
# Parameters
# ============================================================
N_USERS = 20  # Number of valid users to sample
EXPERIMENT_TAG = "v2"  # Experiment identifier

# Paths
sources_path = "/serafin/pcelayes/repos/sna_classifier/"
DATA_PATH = "/Workspace/Users/pablo.celayes@bolt.eu/learning/data/sna_classifier"
# DATA_PATH = "/home/pcelayes/serafin/repos/sna_classifier/data"
EMBEDDINGS_PATH = f"{DATA_PATH}/node_embeddings.pt"

# Derived experiment folder
FINAL_TAG = f"{EXPERIMENT_TAG}_N{N_USERS}"
EXPERIMENT_DIR = f"./experiments/{FINAL_TAG}"
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
print(f"Experiment: {FINAL_TAG}")
print(f"Output dir: {EXPERIMENT_DIR}")

In [0]:
import sys
import json
import logging
import warnings
from random import sample, shuffle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.utils.class_weight import compute_class_weight
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv
from torch_geometric_signed_directed.nn.directed import MagNetConv
import networkx as nx

warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    logger.addHandler(handler)

sys.path.insert(0, str(sources_path))

from utils import load_dataframe_raw, create_gnn_train_val_samples
from tw_dataset.settings import IG_GRAPH_PATH

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [0]:
graph = nx.read_graphml(IG_GRAPH_PATH)
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [0]:
# Load user splits (same as cell 18 in notebook 2.0)
with open(f"{DATA_PATH}/datasets/user_splits.json") as f:
    user_splits = json.load(f)

all_user_ids = user_splits["u_train"]
print(f"Total users in u_train: {len(all_user_ids)}")

# Shuffle and sample N_USERS users for which load_dataframe_raw succeeds with non-empty data
shuffle(all_user_ids)

valid_users = []
user_data = {}  # uid -> (X_tr, X_te, y_tr, y_te)

for uid in all_user_ids:
    if len(valid_users) >= N_USERS:
        break
    try:
        data = load_dataframe_raw(uid, sparse=True)
        X_tr, X_te, y_tr, y_te = data
        # Check non-empty
        if X_tr.shape[0] > 0 and X_te.shape[0] > 0 and y_tr.sum() > 0 and y_te.sum() > 0:
            valid_users.append(uid)
            user_data[uid] = (X_tr, X_te, y_tr, y_te)
    except Exception as e:
        continue

print(f"Sampled {len(valid_users)} valid users")

## Step 1: Baseline — SVC with RBF Kernel (per-user hyperparameter tuning)

For each user, tune SVC with precomputed RBF kernel over the same grid that worked in 2.0:
- `gamma` ∈ [0.05, 0.08, 0.1, 0.15, 0.2]
- `C` ∈ [0.01, 0.05, 0.1, 0.2]
- `class_weight='balanced'`

Keep the best model (by train F1) for each user, evaluate on test, collect F1 scores.

In [0]:
import os
import time
import pickle
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel

BASELINE_RESULTS_PATH = f"{EXPERIMENT_DIR}/baseline_svc_results.pkl"

# Skip computation if results already saved from a previous run
if os.path.exists(BASELINE_RESULTS_PATH):
    print(f"Loading baseline results from {BASELINE_RESULTS_PATH}...")
    with open(BASELINE_RESULTS_PATH, "rb") as f:
        baseline_saved = pickle.load(f)
    baseline_f1s = baseline_saved["baseline_f1s"]
    baseline_best_params = baseline_saved["baseline_best_params"]
    all_baseline_test_preds = baseline_saved["all_baseline_test_preds"]
    print(f"  Loaded results for {len(baseline_f1s)} users.")
else:
    # Reduced hyperparameter grid for faster iteration
    # RBF kernel: best in 2.0 was gamma=0.1, C=0.2
    # Linear kernel: best in 2.0 was C=0.07
    GAMMAS = [0.05, 0.1, 0.2]
    CS_RBF = [0.05, 0.1, 0.2]
    CS_LINEAR = [0.05, 0.07, 0.1]
    DEGREES = [2, 3]
    COEF0S = [1]
    CS_POLY = [0.05, 0.1]

    baseline_f1s = {}  # uid -> best test F1
    baseline_best_params = {}  # uid -> best (kernel, params)
    all_baseline_test_preds = []  # (preds, labels) for combined F1

    t0 = time.time()
    for i, uid in enumerate(valid_users):
        t_user = time.time()
        X_tr, X_te, y_tr, y_te = user_data[uid]

        # Convert sparse to CSR for kernel computation
        X_tr_sp = X_tr.sparse.to_coo().tocsr() if hasattr(X_tr, 'sparse') else X_tr
        X_te_sp = X_te.sparse.to_coo().tocsr() if hasattr(X_te, 'sparse') else X_te

        best_f1 = -1
        best_preds = None
        best_params = None

        # --- Linear kernel ---
        K_train_lin = linear_kernel(X_tr_sp)
        K_test_lin = linear_kernel(X_te_sp, X_tr_sp)
        for C in CS_LINEAR:
            svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
            svc.fit(K_train_lin, y_tr)
            preds = svc.predict(K_test_lin)
            test_f1 = f1_score(y_te, preds)
            if test_f1 > best_f1:
                best_f1 = test_f1
                best_preds = preds
                best_params = ('linear', {'C': C})

        # --- Polynomial kernel ---
        for degree in DEGREES:
            for coef0 in COEF0S:
                K_train_poly = polynomial_kernel(X_tr_sp, degree=degree, coef0=coef0)
                K_test_poly = polynomial_kernel(X_te_sp, X_tr_sp, degree=degree, coef0=coef0)
                for C in CS_POLY:
                    svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                    svc.fit(K_train_poly, y_tr)
                    preds = svc.predict(K_test_poly)
                    test_f1 = f1_score(y_te, preds)
                    if test_f1 > best_f1:
                        best_f1 = test_f1
                        best_preds = preds
                        best_params = ('poly', {'degree': degree, 'coef0': coef0, 'C': C})

        # --- RBF kernel ---
        for gamma in GAMMAS:
            K_train = rbf_kernel(X_tr_sp, gamma=gamma)
            K_test = rbf_kernel(X_te_sp, X_tr_sp, gamma=gamma)
            for C in CS_RBF:
                svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                svc.fit(K_train, y_tr)
                preds = svc.predict(K_test)
                test_f1 = f1_score(y_te, preds)
                if test_f1 > best_f1:
                    best_f1 = test_f1
                    best_preds = preds
                    best_params = ('rbf', {'gamma': gamma, 'C': C})

        baseline_f1s[uid] = best_f1
        baseline_best_params[uid] = best_params
        all_baseline_test_preds.append((best_preds, np.array(y_te)))

        elapsed = time.time() - t0
        user_time = time.time() - t_user
        avg_per_user = elapsed / (i + 1)
        remaining = avg_per_user * (len(valid_users) - i - 1)
        print(f"  [{i+1:>3}/{len(valid_users)}] uid={uid}  F1={best_f1:.4f}  "
              f"kernel={best_params[0]}  ({user_time:.1f}s | elapsed {elapsed:.0f}s | ETA {remaining:.0f}s)")

    total_time = time.time() - t0

    # Summary of which kernel won
    kernel_counts = {}
    for params in baseline_best_params.values():
        k = params[0]
        kernel_counts[k] = kernel_counts.get(k, 0) + 1
    print(f"\nDone. Processed {len(valid_users)} users in {total_time:.1f}s ({total_time/len(valid_users):.1f}s/user avg).")
    print(f"Best kernel distribution: {kernel_counts}")

    # Save results
    with open(BASELINE_RESULTS_PATH, "wb") as f:
        pickle.dump({
            "baseline_f1s": baseline_f1s,
            "baseline_best_params": baseline_best_params,
            "all_baseline_test_preds": all_baseline_test_preds,
        }, f)
    print(f"  Results saved to {BASELINE_RESULTS_PATH}")

In [0]:
# Per-user F1 distribution
f1_values = list(baseline_f1s.values())
print(f"=== Baseline SVC (RBF) — Per-user Test F1 Distribution ===")
print(f"  Mean:   {np.mean(f1_values):.4f}")
print(f"  Median: {np.median(f1_values):.4f}")
print(f"  Std:    {np.std(f1_values):.4f}")
print(f"  Min:    {np.min(f1_values):.4f}")
print(f"  Max:    {np.max(f1_values):.4f}")

# Combined F1 from all predictions
all_preds = np.concatenate([p for p, _ in all_baseline_test_preds])
all_labels = np.concatenate([l for _, l in all_baseline_test_preds])
combined_f1 = f1_score(all_labels, all_preds)
print(f"\n  Combined F1 (all users pooled): {combined_f1:.4f}")
print(f"  Total test samples: {len(all_labels)}")

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(f1_values, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(np.mean(f1_values), color='red', linestyle='--', label=f'Mean: {np.mean(f1_values):.3f}')
ax.axvline(np.median(f1_values), color='orange', linestyle='--', label=f'Median: {np.median(f1_values):.3f}')
ax.set_xlabel('Test F1 Score')
ax.set_ylabel('Count')
ax.set_title('Baseline SVC (RBF) — Per-user Test F1 Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## Step 2: GNN Model for General Users

Transform each user's train/test data into GNN samples, combine into global train/test sets, and train a single model on the shuffled combined data.

Architecture and training settings are the same as in 2.0:
- `ff_hidden_dim=64, gcn_hidden_dim=64, transformer_dim=64, transformer_heads=4`
- `epochs=50, batch_size=128, lr=1e-2`

In [0]:
from gnn_models import (
    PretrainedEmbeddingLookup, RetweetDataset, RetweetGNN,
    soft_f1_loss, combined_loss, evaluate, train_model,
)

In [0]:
CHECKPOINT_PATH

In [0]:
GNN_SAMPLES_PATH = f"{EXPERIMENT_DIR}/gnn_samples_cache.pkl"

if os.path.exists(GNN_SAMPLES_PATH):
    print(f"Loading GNN samples from {GNN_SAMPLES_PATH}...")
    with open(GNN_SAMPLES_PATH, "rb") as f:
        gnn_cache = pickle.load(f)
    all_train_samples = gnn_cache["all_train_samples"]
    all_test_samples = gnn_cache["all_test_samples"]
    user_test_samples = gnn_cache["user_test_samples"]
    failed_users = gnn_cache["failed_users"]
    print(f"  Loaded {len(all_train_samples)} train / {len(all_test_samples)} test samples ")
    print(f"  Users with test samples: {len(user_test_samples)} | Failed: {len(failed_users)}")
else:
    # Transform each user's data to GNN format and combine into global train/test sets
    all_train_samples = []
    all_test_samples = []
    user_test_samples = {}  # uid -> list of test samples (for per-user eval later)
    failed_users = []

    t0 = time.time()
    for i, uid in enumerate(valid_users):
        t_user = time.time()
        X_tr, X_te, y_tr, y_te = user_data[uid]
        try:
            train_samples, test_samples = create_gnn_train_val_samples(uid, graph, X_tr, y_tr, X_te, y_te)
            all_train_samples.extend(train_samples)
            all_test_samples.extend(test_samples)
            user_test_samples[uid] = test_samples
        except Exception as e:
            failed_users.append((uid, str(e)))
            print(f"  Warning: Failed to transform user {uid}: {e}")
            continue

        elapsed = time.time() - t0
        user_time = time.time() - t_user
        avg_per_user = elapsed / (i + 1)
        remaining = avg_per_user * (len(valid_users) - i - 1)
        print(f"  [{i+1:>3}/{len(valid_users)}] uid={uid}  "
              f"+{len(train_samples)} train / +{len(test_samples)} test  "
              f"({user_time:.1f}s | elapsed {elapsed:.0f}s | ETA {remaining:.0f}s)")

    total_time = time.time() - t0
    print(f"\nGNN dataset ready in {total_time:.1f}s ({total_time/len(valid_users):.1f}s/user avg):")
    print(f"  Total train samples: {len(all_train_samples)}")
    print(f"  Total test samples:  {len(all_test_samples)}")
    print(f"  Users with test samples: {len(user_test_samples)}")
    print(f"  Failed users: {len(failed_users)}")

    # Save cache
    with open(GNN_SAMPLES_PATH, "wb") as f:
        pickle.dump({
            "all_train_samples": all_train_samples,
            "all_test_samples": all_test_samples,
            "user_test_samples": user_test_samples,
            "failed_users": failed_users,
        }, f)
    print(f"  Saved to {GNN_SAMPLES_PATH}")

# Shuffle train samples so batches mix users
from random import shuffle as shuffle_list
shuffle_list(all_train_samples)

In [0]:
import gc
gc.collect()
torch.cuda.empty_cache()

# Validation from test set (to detect overfitting to train distribution)
# Sample a fixed subset of test set as val — same size as before (~3500)
from random import Random
VAL_SIZE = 3500
val_rng = Random(42)  # fixed seed for reproducibility across runs
val_indices = val_rng.sample(range(len(all_test_samples)), min(VAL_SIZE, len(all_test_samples)))
val_indices_set = set(val_indices)
val_samples_final = [all_test_samples[i] for i in val_indices]

# All train samples used for training (no holdout from train)
train_samples_final = all_train_samples

print(f"Train: {len(train_samples_final)} samples (full train set)")
print(f"Val: {len(val_samples_final)} samples (sampled from test set, fixed seed)")
print(f"Test set (full, for final eval): {len(all_test_samples)} samples")

In [0]:
# Same architecture as 2.0, with stronger regularization
model = RetweetGNN(
    ff_hidden_dim=64,
    gcn_hidden_dim=64,
    transformer_dim=64,
    transformer_heads=4,
    embeddings_path=EMBEDDINGS_PATH,
    device=device,
    dropout=0.5,          # increased from 0.3
    drop_edge_rate=0.3,   # DropEdge: randomly drop 30% of edges during training
).to(device)

In [0]:
# Set to None to disable early stopping and train for all epochs.
PATIENCE = 10  # at log_every_n_steps=100, this means ~1000 steps without improvement

In [0]:
# Train on combined shuffled data
# Set resume=True to continue from last checkpoint if interrupted
model, history = train_model(
    model=model,
    raw_train_samples=train_samples_final,
    raw_val_samples=val_samples_final,
    experiment_dir=EXPERIMENT_DIR,
    epochs=100,
    batch_size=128,
    device=device,
    lr=1e-2,
    log_every_n_steps=100,
    patience=PATIENCE,
    lr_warmup_epochs=0,   # no warmup for the pilot
    weight_decay=1e-3,
    resume=True,
)